In [1]:
# Step 1: install packages
!pip install -q transformers sentencepiece scikit-learn pandas numpy matplotlib shap

In [2]:
# Step 1: imports
import os
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

In [3]:
print("Python executable:", os.sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True

Python executable: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA available: True
Torch CUDA version: 12.8
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Using device: cuda:0


In [4]:
# Step 3: load dataset
csv_path = "/kaggle/input/datasets/dhivyeshrk/diseases-and-symptoms-dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv"
df = pd.read_csv(csv_path)

print("Original shape:", df.shape)
print(df.head())
print(df.columns.tolist())

Original shape: (246945, 378)
         diseases  anxiety and nervousness  depression  shortness of breath  \
0  panic disorder                        1           0                    1   
1  panic disorder                        0           0                    1   
2  panic disorder                        1           1                    1   
3  panic disorder                        1           0                    0   
4  panic disorder                        1           1                    0   

   depressive or psychotic symptoms  sharp chest pain  dizziness  insomnia  \
0                                 1                 0          0         0   
1                                 1                 0          1         1   
2                                 1                 0          1         1   
3                                 1                 0          1         1   
4                                 0                 0          0         1   

   abnormal involuntary mo

In [5]:
# Step 4: convert 0/1 symptom columns into text
label_col = "diseases"
symptom_cols = [col for col in df.columns if col != label_col]

def row_to_text(row):
    active_symptoms = [symptom for symptom in symptom_cols if row[symptom] == 1]
    return " [SEP] ".join(active_symptoms)

df_model = pd.DataFrame({
    "text": df.apply(row_to_text, axis=1),
    "diseases": df[label_col].astype(str)
}).copy()

print("New shape:", df_model.shape)
print(df_model.head())
print(df_model["text"].iloc[0])

New shape: (246945, 2)
                                                text        diseases
0  anxiety and nervousness [SEP] shortness of bre...  panic disorder
1  shortness of breath [SEP] depressive or psycho...  panic disorder
2  anxiety and nervousness [SEP] depression [SEP]...  panic disorder
3  anxiety and nervousness [SEP] depressive or ps...  panic disorder
4  anxiety and nervousness [SEP] depression [SEP]...  panic disorder
anxiety and nervousness [SEP] shortness of breath [SEP] depressive or psychotic symptoms [SEP] chest tightness [SEP] palpitations [SEP] irregular heartbeat [SEP] breathing fast


In [6]:
print(df_model["diseases"].nunique())
print(df_model["diseases"].value_counts().head())

773
diseases
cystitis                          1219
vulvodynia                        1218
nose disorder                     1218
complex regional pain syndrome    1217
spondylosis                       1216
Name: count, dtype: int64


In [7]:
# Step 5: remove very rare classes, encode labels, split data
class_counts = df_model["diseases"].value_counts()

valid_classes = class_counts[class_counts >= 2].index
df_model = df_model[df_model["diseases"].isin(valid_classes)].copy()

print("Filtered shape:", df_model.shape)
print("Remaining classes:", df_model["diseases"].nunique())

label_encoder = LabelEncoder()
df_model["label"] = label_encoder.fit_transform(df_model["diseases"])

num_labels = len(label_encoder.classes_)
print("Number of classes after filtering:", num_labels)

X_train, X_test, y_train, y_test = train_test_split(
    df_model["text"].tolist(),
    df_model["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_model["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Sample label id:", y_train[0])
print("Sample label name:", label_encoder.inverse_transform([y_train[0]])[0])

Filtered shape: (246926, 2)
Remaining classes: 754
Number of classes after filtering: 754
Train size: 197540
Test size: 49386
Sample label id: 172
Sample label name: dental caries


In [8]:
print(type(X_train[0]))
print(X_train[0][:300])

<class 'str'>
toothache [SEP] peripheral edema [SEP] mouth pain [SEP] pain in gums


In [9]:
# Step 6: tokenizer and tokenization
model_name = "nghuyong/ernie-2.0-en"
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizing train data...")
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="pt"
)

print("Tokenizing test data...")
test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="pt"
)

print("Train input_ids shape:", train_encodings["input_ids"].shape)
print("Test input_ids shape:", test_encodings["input_ids"].shape)

config.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizing train data...
Tokenizing test data...
Train input_ids shape: torch.Size([197540, 68])
Test input_ids shape: torch.Size([49386, 63])


In [10]:
print(tokenizer.decode(train_encodings["input_ids"][0][:50]))

[CLS] toothache [SEP] peripheral edema [SEP] mouth pain [SEP] pain in gums [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [11]:
# Step 7: dataset and dataloader
class EncodedDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = EncodedDataset(train_encodings, y_train)
test_dataset = EncodedDataset(test_encodings, y_test)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 3087
Test batches: 772


In [12]:
# Step 8: model + DataParallel
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

base_model = base_model.to(device)

if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = nn.DataParallel(base_model)
else:
    model = base_model

print("Model device:", next(model.parameters()).device)

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ErnieForSequenceClassification LOAD REPORT from: nghuyong/ernie-2.0-en
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using 2 GPUs with DataParallel
Model device: cuda:0


In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print("Optimizer and loss created successfully")

Optimizer and loss created successfully


In [14]:
# Step 9: training and evaluation functions with clean epoch results
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc, all_labels, all_preds

In [15]:
epochs = 2
total_start_time = time.time()

for epoch in range(epochs):
    epoch_start = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    epoch_time = time.time() - epoch_start

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss = {train_loss:.4f}, "
        f"Train Acc = {train_acc:.4f}, "
        f"Epoch Time = {epoch_time:.2f} sec"
    )

total_training_time = time.time() - total_start_time
print(f"\nTotal Training Time: {total_training_time:.2f} seconds")

Epoch 1: Train Loss = 2.4577, Train Acc = 0.5894, Epoch Time = 1545.01 sec
Epoch 2: Train Loss = 0.7469, Train Acc = 0.8184, Epoch Time = 1551.58 sec

Total Training Time: 3096.60 seconds


In [16]:
test_loss, test_acc, y_true, y_pred = evaluate(
    model, test_loader, criterion, device
)

print("\nFinal Test Results")
print(f"Test Loss = {test_loss:.4f}")
print(f"Test Accuracy = {test_acc:.4f}")


Final Test Results
Test Loss = 0.5771
Test Accuracy = 0.8371


In [17]:
from sklearn.metrics import classification_report

all_labels = list(range(num_labels))

print("\nClassification Report:\n")
report = classification_report(
    y_true,
    y_pred,
    labels=all_labels,
    target_names=label_encoder.classes_,
    zero_division=0
)

print(report)


Classification Report:

                                                          precision    recall  f1-score   support

                               abdominal aortic aneurysm       1.00      0.96      0.98        28
                                        abdominal hernia       0.94      1.00      0.97        81
                                         abscess of nose       0.75      0.88      0.81        58
                                     abscess of the lung       0.00      0.00      0.00         4
                                  abscess of the pharynx       0.67      0.91      0.77        68
                                    acanthosis nigricans       1.00      1.00      1.00         6
                                               acariasis       0.50      0.57      0.53         7
                                               achalasia       0.80      0.47      0.59        17
                                                    acne       0.81      0.74      0.77     

In [18]:
def predict_proba(texts):
    model.eval()

    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1)

    return probs.detach().cpu().numpy()

In [19]:
import shap

explainer = shap.Explainer(predict_proba, tokenizer)
print("SHAP explainer ready")

SHAP explainer ready


In [20]:
sample_texts = [
    X_test[0],
    X_test[1],
    X_test[2]
]

In [21]:
shap_values = explainer(sample_texts)

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 4it [00:13,  6.98s/it]               


In [22]:
shap_values.output_names = list(label_encoder.classes_)

In [23]:
import numpy as np

for i, text in enumerate(sample_texts):
    probs = predict_proba([text])[0]
    pred_id = int(np.argmax(probs))
    pred_name = label_encoder.inverse_transform([pred_id])[0]

    print("\n==============================")
    print("Text:", text)
    print("Predicted Disease:", pred_name)
    print("==============================")

    shap.plots.text(shap_values[i, :, pred_id])


Text: cough [SEP] nasal congestion [SEP] vomiting [SEP] decreased appetite [SEP] fever [SEP] ache all over [SEP] skin rash
Predicted Disease: strep throat



Text: diminished hearing [SEP] facial pain [SEP] ear pain [SEP] plugged feeling in ear [SEP] itchy ear(s) [SEP] fever
Predicted Disease: otitis externa (swimmer's ear)



Text: double vision [SEP] lacrimation [SEP] blindness
Predicted Disease: cataract


In [24]:
probs = predict_proba([text])[0]
top3 = np.argsort(probs)[-3:][::-1]

for idx in top3:
    print(label_encoder.inverse_transform([idx])[0], probs[idx])

cataract 0.6069631
retinal detachment 0.2522929
macular degeneration 0.040683076


In [25]:
shap.plots.text(shap_values[0, :, pred_id])

In [26]:
sample_text = X_test[0]

# get prediction
probs = predict_proba([sample_text])[0]
pred_id = int(np.argmax(probs))
pred_name = label_encoder.inverse_transform([pred_id])[0]

print("Text:", sample_text)
print("Predicted:", pred_name)
print("Probability:", probs[pred_id])

# SHAP
shap_values = explainer([sample_text])
shap_values.output_names = list(label_encoder.classes_)

shap.plots.text(shap_values[0, :, pred_id])

Text: cough [SEP] nasal congestion [SEP] vomiting [SEP] decreased appetite [SEP] fever [SEP] ache all over [SEP] skin rash
Predicted: strep throat
Probability: 0.9794519


In [27]:
# define sample
sample_text = X_test[0]

# prediction
probs = predict_proba([sample_text])[0]

pred_id = int(np.argmax(probs))
pred_name = label_encoder.inverse_transform([pred_id])[0]

print("Text:", sample_text)
print("Predicted:", pred_name)
print("Probability:", probs[pred_id])

# SHAP
shap_values = explainer([sample_text])
shap_values.output_names = list(label_encoder.classes_)

shap.plots.text(shap_values[0, :, pred_id])

Text: cough [SEP] nasal congestion [SEP] vomiting [SEP] decreased appetite [SEP] fever [SEP] ache all over [SEP] skin rash
Predicted: strep throat
Probability: 0.9794519
